In [35]:
import matplotlib
matplotlib.use("Agg")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (14, 5), "axes.titlesize": 14})

PLOT_DIR = "eda_plots"
os.makedirs(PLOT_DIR, exist_ok=True)

def savefig(name):
    plt.savefig(os.path.join(PLOT_DIR, name), dpi=100, bbox_inches="tight")
    plt.close()
    print(f"   saved {PLOT_DIR}/{name}")

In [36]:
df = pd.read_csv("data/sensor_raw.csv")
print(df.head())
print("=" * 40)
print(f"Rows: {df.size}")
print(f"Cols: {df.columns}")
print(f"Missing: {(df.isnull().sum())}")





   Class  DriverID  TaskID  Hour  Minute  Second     GyroX     GyroY  \
0      3  Driver-1       1    15      39      39 -0.496183  2.511450   
1      3  Driver-1       1    15      39      40 -1.519084  2.832061   
2      3  Driver-1       1    15      39      41  0.679389  3.206107   
3      3  Driver-1       1    15      39      42  0.053435  9.244275   
4      3  Driver-1       1    15      39      42 -0.664122  8.030534   

      GyroZ      AccX      AccY      AccZ  
0  0.702290  0.226807 -0.132324 -1.010498  
1 -1.244275  0.331299 -0.145752 -0.998291  
2 -0.229008  0.431396 -0.209717 -1.004150  
3  2.893130  0.370117 -0.062012 -0.901123  
4  0.961832  0.364990 -0.132324 -1.001221  
Rows: 13368
Cols: Index(['Class', 'DriverID', 'TaskID', 'Hour', 'Minute', 'Second', 'GyroX',
       'GyroY', 'GyroZ', 'AccX', 'AccY', 'AccZ'],
      dtype='str')
Missing: Class       0
DriverID    0
TaskID      0
Hour        0
Minute      0
Second      0
GyroX       0
GyroY       0
GyroZ       0
AccX  

In [37]:
samples_per_second = df.groupby(['Hour', 'Minute', 'Second']).size()
print("Samples per second distribution:")
print(samples_per_second.value_counts())
estimated_hz = samples_per_second.mode()[0]
print(f"\nEstimated Sampling Rate: {estimated_hz} Hz")

Samples per second distribution:
1    571
2    193
3     34
4      7
5      3
6      2
Name: count, dtype: int64

Estimated Sampling Rate: 1 Hz


In [38]:
# Number of driving events
df.groupby(['Class']).size()

Class
1    252
2    288
3    350
4    224
dtype: int64

In [39]:
plt.figure(figsize=(20,10))
plt.plot(df['AccX'].iloc[:100])

In [40]:
#Schema inspection
print("\n— dtypes —")
print(df.dtypes)
print(f"\n— Memory usage —\n{df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
print(f"\n— Missing values —\n{df.isnull().sum()}")
print(f"\n— Duplicate rows: {df.duplicated().sum()}")
print(f"\n— Describe —")
print(df.describe())


— dtypes —
Class         int64
DriverID        str
TaskID        int64
Hour          int64
Minute        int64
Second        int64
GyroX       float64
GyroY       float64
GyroZ       float64
AccX        float64
AccY        float64
AccZ        float64
dtype: object

— Memory usage —
0.16 MB

— Missing values —
Class       0
DriverID    0
TaskID      0
Hour        0
Minute      0
Second      0
GyroX       0
GyroY       0
GyroZ       0
AccX        0
AccY        0
AccZ        0
dtype: int64

— Duplicate rows: 0

— Describe —
             Class       TaskID         Hour       Minute       Second  \
count  1114.000000  1114.000000  1114.000000  1114.000000  1114.000000   
mean      2.490126    14.039497    14.138241    27.129264    27.045781   
std       1.051415     8.071906     1.151861    13.918108    16.892894   
min       1.000000     1.000000    12.000000     0.000000     0.000000   
25%       2.000000     7.000000    14.000000    18.000000    13.000000   
50%       3.000000    14.000

In [41]:
samples_per_second = df.groupby(["Hour", "Minute", "Second"]).size()

print("\nSamples-per-second distribution:")
print(samples_per_second.value_counts().sort_index())

estimated_hz = int(samples_per_second.mode()[0])
print(f"\n>>> Estimated sampling rate: {estimated_hz} Hz")

fig, ax = plt.subplots()
samples_per_second.hist(bins=30, ax=ax, edgecolor="black")
ax.set_xlabel("Samples per second")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Samples per Wall-Clock Second")
ax.axvline(estimated_hz, color="red", ls="--", label=f"Mode = {estimated_hz} Hz")
ax.legend()
plt.tight_layout()
savefig("01_sampling_rate.png")


Samples-per-second distribution:
1    571
2    193
3     34
4      7
5      3
6      2
Name: count, dtype: int64

>>> Estimated sampling rate: 1 Hz
   saved eda_plots/01_sampling_rate.png


In [42]:
class_counts = df["Class"].value_counts().sort_index()
print(f"\nClass counts:\n{class_counts}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_counts.plot.bar(ax=axes[0], edgecolor="black")
axes[0].set_title("Driving-Event Class Counts")
axes[0].set_ylabel("Samples")
axes[0].set_xlabel("Class")
axes[0].tick_params(axis="x", rotation=0)

class_counts.plot.pie(ax=axes[1], autopct="%1.1f%%", startangle=140)
axes[1].set_ylabel("")
axes[1].set_title("Class Proportions")

plt.tight_layout()
savefig("02_class_distribution.png")

imbalance_ratio = class_counts.max() / class_counts.min()
print(f"\nImbalance ratio (max / min): {imbalance_ratio:.1f}x")


Class counts:
Class
1    252
2    288
3    350
4    224
Name: count, dtype: int64


   saved eda_plots/02_class_distribution.png

Imbalance ratio (max / min): 1.6x


In [43]:
META_COLS = ["Class", "DriverID", "TaskID", "Hour", "Minute", "Second"]
LABEL_COL = "Class"
sensor_cols = [c for c in df.columns if c not in META_COLS]
print(f"\nSensor channels ({len(sensor_cols)}): {sensor_cols}")
print(df[sensor_cols].describe().T)

# %% Per-channel histograms
n = len(sensor_cols)
ncols = min(n, 3)
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = np.atleast_2d(axes)

for idx, col in enumerate(sensor_cols):
    ax = axes[idx // ncols, idx % ncols]
    df[col].hist(bins=80, ax=ax, edgecolor="black", alpha=0.7)
    ax.set_title(col)
    ax.set_xlabel("Value")

for idx in range(n, nrows * ncols):
    axes[idx // ncols, idx % ncols].set_visible(False)

fig.suptitle("Sensor-Channel Value Distributions", y=1.02, fontsize=15)
plt.tight_layout()
plt.show()
savefig("03_channel_histograms.png")



Sensor channels (6): ['GyroX', 'GyroY', 'GyroZ', 'AccX', 'AccY', 'AccZ']
        count      mean        std        min       25%       50%       75%  \
GyroX  1114.0 -0.829957   3.404297 -14.946565 -2.274809 -0.698473  0.784351   
GyroY  1114.0  4.122309   3.252790 -10.351145  2.610687  4.290076  5.761450   
GyroZ  1114.0  1.076404  12.049384 -50.259542  0.282443  0.961832  1.992366   
AccX   1114.0  0.252417   0.183155  -0.252686  0.132202  0.244507  0.372559   
AccY   1114.0 -0.095110   0.190407  -0.793457 -0.179077 -0.102295 -0.023010   
AccZ   1114.0 -0.983215   0.098061  -1.367920 -1.031372 -0.986938 -0.928223   

             max  
GyroX  12.778626  
GyroY  16.793893  
GyroZ  45.442748  
AccX    0.747803  
AccY    0.768555  
AccZ   -0.456787  
   saved eda_plots/03_channel_histograms.png


/var/folders/6y/w83434vd58jf6rqmf82sh3yc0000gn/T/ipykernel_99843/703156852.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [44]:
#Raw waveform plot (first 500 samples)
PREVIEW_LEN = 500

fig, axes = plt.subplots(len(sensor_cols), 1,
                         figsize=(16, 3 * len(sensor_cols)), sharex=True)
if len(sensor_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, sensor_cols):
    ax.plot(df[col].iloc[:PREVIEW_LEN], linewidth=0.6)
    ax.set_ylabel(col)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Sample index")
fig.suptitle(f"Raw Waveforms (first {PREVIEW_LEN} samples)", fontsize=15)
plt.tight_layout()
savefig("04_raw_waveforms.png")
plt.show()


   saved eda_plots/04_raw_waveforms.png


/var/folders/6y/w83434vd58jf6rqmf82sh3yc0000gn/T/ipykernel_99843/4094686070.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [45]:
# Per class signals
PRIMARY_AXIS = "GyroX"
classes = sorted(df[LABEL_COL].unique())

fig, axes = plt.subplots(len(classes), 1,
                         figsize=(16, 3 * len(classes)), sharex=True, sharey=True)
if len(classes) == 1:
    axes = [axes]

for ax, cls in zip(axes, classes):
    subset = df.loc[df[LABEL_COL] == cls, PRIMARY_AXIS].values
    ax.plot(subset[:PREVIEW_LEN], linewidth=0.6)
    ax.set_ylabel(PRIMARY_AXIS)
    ax.set_title(f"Class: {cls}  (n={len(subset):,})")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Sample index")
fig.suptitle(f"{PRIMARY_AXIS} Waveform by Driving-Event Class", fontsize=15)
plt.tight_layout()
savefig("05_per_class_GyroX.png")

   saved eda_plots/05_per_class_GyroX.png


In [46]:
# Per-class waveform overlay for AccX
PRIMARY_AXIS2 = "AccX"

fig, axes = plt.subplots(len(classes), 1,
                         figsize=(16, 3 * len(classes)), sharex=True, sharey=True)
if len(classes) == 1:
    axes = [axes]

for ax, cls in zip(axes, classes):
    subset = df.loc[df[LABEL_COL] == cls, PRIMARY_AXIS2].values
    ax.plot(subset[:PREVIEW_LEN], linewidth=0.6)
    ax.set_ylabel(PRIMARY_AXIS2)
    ax.set_title(f"Class: {cls}  (n={len(subset):,})")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Sample index")
fig.suptitle(f"{PRIMARY_AXIS2} Waveform by Driving-Event Class", fontsize=15)
plt.tight_layout()
savefig("06_per_class_AccX.png")

   saved eda_plots/06_per_class_AccX.png


In [56]:
# Correlation & Cross-Channel Relationships

corr = df[sensor_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, ax=ax)
ax.set_title("Sensor-Channel Correlation Matrix")
plt.tight_layout()
savefig("07_correlation_heatmap.png")

# %% Pairplot — use Acc vs Gyro subset (4 channels) for speed

pair_cols = ["AccX", "AccZ", "GyroX", "GyroY"]
label_elaborate = {1: 'Acceleration', 2: 'Right', 3: 'Left', 4: 'Brake'}
LABEL_NAME = 'label_name'
df['label_name'] = df[LABEL_COL].replace(label_elaborate)

pair_df = df[pair_cols + [LABEL_NAME]].sample(min(1000, len(df)), random_state=0)

g = sns.pairplot(pair_df, hue=LABEL_NAME, plot_kws={"alpha": 0.5, "s": 20})
g.figure.suptitle("Pairwise Sensor Scatter by Class (subset)", y=1.02)
savefig("08_pairplot.png")

   saved eda_plots/07_correlation_heatmap.png
   saved eda_plots/08_pairplot.png


In [51]:
df['label_name']

0       3
1       3
2       3
3       3
4       3
       ..
1109    4
1110    4
1111    4
1112    4
1113    4
Name: label_name, Length: 1114, dtype: int64

In [48]:
# Z-score outlier counts
z_scores = np.abs(stats.zscore(df[sensor_cols], nan_policy="omit"))
outlier_mask = z_scores > 3

print("\nOutlier counts (|z| > 3) per channel:")
print(pd.Series(outlier_mask.sum(axis=0), index=sensor_cols))
print(f"\nTotal rows with ≥1 outlier: {outlier_mask.any(axis=1).sum()} "
      f"({outlier_mask.any(axis=1).mean():.2%})")

# %% Box-plots per class
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, col in zip(axes, sensor_cols):
    sns.boxplot(data=df, x=LABEL_COL, y=col, ax=ax)
    ax.set_title(col)

fig.suptitle("Sensor Distributions by Class (box-plots)", y=1.02, fontsize=15)
plt.tight_layout()
savefig("09_boxplots_by_class.png")



Outlier counts (|z| > 3) per channel:
GyroX    20
GyroY    14
GyroZ    51
AccX      0
AccY     21
AccZ     15
dtype: int64

Total rows with ≥1 outlier: 89 (7.99%)
   saved eda_plots/09_boxplots_by_class.png


In [49]:
window_seconds = 2
window_size = window_seconds * estimated_hz
overlap_frac = 0.5
stride = max(1, int(window_size * (1 - overlap_frac)))

print(f"\nSampling rate  : {estimated_hz} Hz")
print(f"Window length  : {window_seconds}s  →  {window_size} samples")
print(f"Overlap        : {overlap_frac:.0%}  →  stride = {stride} samples")
print(f"Input shape    : ({window_size}, {len(sensor_cols)})")

n_windows_approx = (len(df) - window_size) // stride + 1
print(f"≈ {n_windows_approx:,} windows from {len(df):,} rows")


Sampling rate  : 1 Hz
Window length  : 2s  →  2 samples
Overlap        : 50%  →  stride = 1 samples
Input shape    : (2, 6)
≈ 1,113 windows from 1,114 rows


In [50]:
# FEature files

feature_dir = "data/Features By Window Size"
feature_files = sorted([f for f in os.listdir(feature_dir) if f.endswith(".csv")])

print(f"\nAvailable feature files ({len(feature_files)}):")
for f in feature_files:
    fpath = os.path.join(feature_dir, f)
    tmp = pd.read_csv(fpath)
    ws = f.split("_")[-1].replace(".csv", "")
    print(f"  {f:30s}  to shape {tmp.shape}, window={ws}")



Available feature files (17):
  features_14.csv                 to shape (1102, 61), window=14
  sero_features_10.csv            to shape (1102, 61), window=10
  sero_features_11.csv            to shape (1102, 61), window=11
  sero_features_12.csv            to shape (1102, 61), window=12
  sero_features_13.csv            to shape (1102, 61), window=13
  sero_features_15.csv            to shape (1102, 61), window=15
  sero_features_16.csv            to shape (1102, 61), window=16
  sero_features_17.csv            to shape (1102, 61), window=17
  sero_features_18.csv            to shape (1102, 61), window=18
  sero_features_19.csv            to shape (1102, 61), window=19
  sero_features_20.csv            to shape (1102, 61), window=20
  sero_features_4.csv             to shape (1102, 61), window=4
  sero_features_5.csv             to shape (1102, 61), window=5
  sero_features_6.csv             to shape (1102, 61), window=6
  sero_features_7.csv             to shape (1102, 61), window=

# 11 — Results & Key Takeaways
| Item | Finding |
|---|---|
| **Rows / Columns** | `df.shape` — see cell above |
| **Sensor channels** | GyroX, GyroY, GyroZ, AccX, AccY, AccZ (6 channels) |
| **Sampling rate** | ~`estimated_hz` Hz |
| **Classes** | 4 driving event classes |
| **Imbalance** | ratio printed above — consider class-weights if > 2x |
| **Missing values** | see cell 1 — handle before windowing |
| **Outliers** | z > 3 counts above — clip or robust-scale |
| **Suggested window** | 2 s @ estimated_hz → shape `(window_size, 6)` |

# **Considerations for the CNN pipeline:**
1. Impute / drop missing values.
2. Normalise each sensor channel (z-score or min-max per channel).
3. Segment into fixed-length windows with the parameters above.
4. Split windows into train / val / test **by contiguous time blocks**
    (not random) to avoid data leakage.
5. Build a 1-D CNN (Conv1D → BatchNorm → ReLU → Pool → … → Dense → Softmax).
6. Use class-weighted cross-entropy if imbalance is significant.